#### Forecast API

In [23]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

def main():
    # Configurar la sesión con caché (1 hora) y reintentos en caso de error
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    # Lista de variables horarias a solicitar
    hourly_vars = [
        "temperature_2m", "dew_point_2m", "relative_humidity_2m", "apparent_temperature", 
        "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "visibility", 
        "evapotranspiration", "et0_fao_evapotranspiration", "vapour_pressure_deficit", "wind_speed_10m", 
        "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m", 
        "wind_direction_120m", "wind_direction_180m", "wind_gusts_10m", "temperature_80m", "temperature_120m", 
        "temperature_180m", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", 
        "soil_temperature_54cm", "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm", 
        "soil_moisture_9_to_27cm", "soil_moisture_27_to_81cm", "uv_index", "uv_index_clear_sky", "is_day", 
        "sunshine_duration", "wet_bulb_temperature_2m", "cape", "lifted_index", "convective_inhibition", 
        "freezing_level_height", "boundary_layer_height", "shortwave_radiation", "diffuse_radiation", 
        "global_tilted_irradiance", "shortwave_radiation_instant", "diffuse_radiation_instant", 
        "global_tilted_irradiance_instant", "direct_radiation", "direct_normal_irradiance", 
        "terrestrial_radiation", "direct_radiation_instant", "direct_normal_irradiance_instant", 
        "terrestrial_radiation_instant", "pressure_msl"
    ]

    # Parámetros para la consulta: 1 día hacia atrás y 1 día de pronóstico
    params = {
        "latitude": 18.2158,
        "longitude": -71.0998,
        "hourly": hourly_vars,  # Se envía la lista completa de variables
        "timezone": "auto",     # En la API se recomienda especificar un timezone válido; "auto" se usa en la web
        "past_days": 1,
        "forecast_days": 7,
        "models": "best_match"
    }

    # Realizar la consulta a la API
    url = "https://api.open-meteo.com/v1/forecast"
    responses = openmeteo.weather_api(url, params=params)
    if not responses:
        print("No se recibió respuesta de la API.")
        return

    # Procesar la primera respuesta (para una única ubicación)
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N, {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone: {response.Timezone()} {response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()} s")

    # Procesar los datos horarios
    hourly = response.Hourly()
    start_time = pd.to_datetime(hourly.Time(), unit="s", utc=True)
    end_time = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True)
    interval = pd.Timedelta(seconds=hourly.Interval())
    
    date_range = pd.date_range(start=start_time, end=end_time, freq=interval, inclusive="left")

    # Construir un diccionario con los datos utilizando el mismo orden que se solicitó
    data_dict = {"date": date_range}
    for idx, var in enumerate(hourly_vars):
        # Se asume que el orden de las variables en la respuesta es el mismo que el de la lista enviada
        values = hourly.Variables(idx).ValuesAsNumpy()
        data_dict[var] = values

    # Crear el DataFrame de pandas
    df_hourly = pd.DataFrame(data=data_dict)
    print(df_hourly.head())
    df_hourly.to_csv('../docs/forecast_api_request.csv')

if __name__ == "__main__":
    main()


Coordinates: 18.25°N, -71.125°E
Elevation: 29.0 m asl
Timezone: b'America/Santo_Domingo' b'GMT-4'
Timezone difference to GMT+0: -14400 s
                       date  temperature_2m  dew_point_2m  \
0 2025-04-02 04:00:00+00:00       24.068001     21.195229   
1 2025-04-02 05:00:00+00:00       23.368000     21.266676   
2 2025-04-02 06:00:00+00:00       22.968000     21.238871   
3 2025-04-02 07:00:00+00:00       22.618000     21.073219   
4 2025-04-02 08:00:00+00:00       22.318001     20.954391   

   relative_humidity_2m  apparent_temperature  surface_pressure  cloud_cover  \
0                  84.0             27.488958       1014.015381          0.0   
1                  88.0             26.995796       1013.608765          0.0   
2                  90.0             26.530325       1013.305298          0.0   
3                  91.0             26.144613       1012.503784          0.0   
4                  92.0             25.939358       1012.201233          7.0   

   cloud_cover_

In [ ]:
import requests, joblib
import pandas as pd, numpy as np
from pathlib import Path

# Paths
CSV_INFO    = Path("../data/lookup/central_info.csv")
MODEL_CLEAN_PATH = Path("../data/interim/meteo_data_with_generation/parque_solar_girasol.parquet")
MODEL_PATH  = Path("../data/models/tft/best_tft_model.pth")
OUTPUT_PATH = Path("../data/processed_predictions/parque_solar_girasol_forecast_predictions.parquet")

# Se cargan las columnas que el modelo espera (excluyendo "generation")
EXPECTED_FEATURES = (
    pd.read_parquet(MODEL_CLEAN_PATH)
      .drop(columns="generation")
      .select_dtypes(include=[np.number])
      .columns.tolist()
)

WEATHER_VARS = [
    "temperature_2m", "wind_speed_10m", "relative_humidity_2m", "wind_gusts_10m",
    "vapour_pressure_deficit", "cloud_cover", "cloud_cover_low", "cloud_cover_mid",
    "cloud_cover_high", "surface_pressure", "pressure_msl", "apparent_temperature",
    "rain", "shortwave_radiation", "diffuse_radiation", "global_tilted_irradiance",
    "shortwave_radiation_instant", "diffuse_radiation_instant", "global_tilted_irradiance_instant",
    "direct_radiation", "direct_normal_irradiance", "terrestrial_radiation",
    "direct_radiation_instant", "direct_normal_irradiance_instant", "terrestrial_radiation_instant"
]

def get_project_coordinates(name, csv_path):
    df = pd.read_csv(csv_path, encoding="latin1")
    row = df[df["CENTRAL"].str.lower() == name.lower()]
    return (float(row.Latitud.iloc[0]), float(row.Longitud.iloc[0])) if not row.empty else None

def get_forecast(lat, lon, days=2):
    """
    Solicita a la API de Open-Meteo datos para 'days' días.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join(WEATHER_VARS),
        "forecast_days": days,
        "timezone": "UTC"
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["hourly"])
    df["date"] = pd.to_datetime(df.pop("time"), utc=True)
    return df.set_index("date")

def add_features(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index, utc=True)
    
    # Características temporales
    df["hour"] = df.index.hour
    df["hour_sin"] = np.sin(2 * np.pi * df.hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df.hour / 24)
    df["day_of_week"] = df.index.dayofweek
    df["dow_sin"] = np.sin(2 * np.pi * df.day_of_week / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df.day_of_week / 7)
    df["month"] = df.index.month
    df["month_sin"] = np.sin(2 * np.pi * df.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df.month / 12)
    
    # Agregar lags y medias móviles para columnas numéricas
    numeric = df.select_dtypes(include=[np.number]).columns
    for col in numeric:
        for lag in (1, 2, 3):
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
        for w in (3, 6):
            df[f"{col}_ma{w}"] = df[col].rolling(window=w, min_periods=1).mean()
    
    # Agregamos generation_diff (se creará pero se eliminará antes de predecir)
    df["generation_diff"] = 0
    return df.dropna()

def main():
    coords = get_project_coordinates("parque solar girasol", Path("../data/lookup/central_info.csv")) or (18.333668, -70.198569)
    
    # Solicitar 2 días: el día anterior (histórico) y el día de pronóstico
    df_all = get_forecast(*coords, days=2)
    # Ajustar 4 horas hacia atrás si se requiere
    df_all.index -= pd.Timedelta(hours=4)
    
    # Separar los días (suponiendo que la API devuelve datos de 2 días distintos)
    unique_days = df_all.index.normalize().unique()
    if len(unique_days) < 2:
        raise ValueError("No se obtuvieron datos de dos días. Verifica la respuesta de la API.")
    hist_day = unique_days[0]
    fcst_day = unique_days[1]
    
    df_hist = df_all.loc[df_all.index.normalize() == hist_day]
    df_fcst = df_all.loc[df_all.index.normalize() == fcst_day]
    
    # Concatenar el día anterior (para calcular lags) con el día de pronóstico
    df_combined = pd.concat([df_hist, df_fcst])
    df_combined_feat = add_features(df_combined)
    
    # Extraer únicamente la parte correspondiente al día de pronóstico
    df_fcst_feat = df_combined_feat.loc[df_fcst.index.min(): df_fcst.index.max()]
    
    # Eliminar la columna "generation_diff" si existe
    df_fcst_feat = df_fcst_feat.drop(columns=["generation_diff"], errors="ignore")
    
    # Rellenar con ceros las columnas faltantes (para que coincida con EXPECTED_FEATURES)
    missing = set(EXPECTED_FEATURES) - set(df_fcst_feat.columns)
    for col in missing:
        df_fcst_feat[col] = 0
    
    # Seleccionar y ordenar las columnas según EXPECTED_FEATURES
    df_final = df_fcst_feat[EXPECTED_FEATURES]
    
    if df_final.empty:
        raise ValueError("El DataFrame final para el pronóstico está vacío. Revisa las ventanas de tiempo y los lags.")
    
    # Reordenar las columnas de acuerdo a lo que esperaba el modelo (si el modelo guarda feature_names_in_)
    model = joblib.load(MODEL_PATH)
    if hasattr(model, "feature_names_in_"):
        df_final = df_final[model.feature_names_in_]
    
    # Predecir
    preds = model.predict(df_final)
    
    # Guardar las predicciones
    pd.DataFrame(preds, index=df_final.index, columns=["prediction"]).to_parquet(OUTPUT_PATH)
    print("✅ Predicciones guardadas en", OUTPUT_PATH)

if __name__ == "__main__":
    main()

C:\Users\ferna\AppData\Local\Temp\ipykernel_9884\2950161193.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_ma{w}"] = df[col].rolling(window=w, min_periods=1).mean()
C:\Users\ferna\AppData\Local\Temp\ipykernel_9884\2950161193.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{lag}"] = df[col].shift(lag)
C:\Users\ferna\AppData\Local\Temp\ipykernel_9884\2950161193.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has p

UnpicklingError: persistent IDs in protocol 0 must be ASCII strings

In [10]:
import pandas as pd
from pathlib import Path
OUTPUT_PATH = Path("../data/interim/post_despacho_transformed.parquet")

# Cargar el DataFrame de predicciones
df_post_despacho_transformed = pd.read_parquet(OUTPUT_PATH)

# Imprimir el DataFrame
df_post_despacho_transformed


,timestamp,aes andres,aguacate 1,aguacate 2,aniana vargas 1,aniana vargas 2,baiguaque 1,baiguaque 2,barahona carbon,bersal,...,tavera 1,tavera 2,total eolico,total generado,total hidroelectrica,total programado,total solar,total termico,valdesia 1,valdesia 2
0,2013-01-01 00:00:00,207.00,25.70,26.70,0.0,0.0,0.2,0.0,43.70,NaN,...,0.00,0.00,11.66,1557.26,167.33,1695.10,NaN,1378.27,0.00,0.0
1,2013-01-01 01:00:00,241.00,25.70,25.70,0.0,0.0,0.2,0.0,43.50,NaN,...,0.00,0.00,20.60,1534.19,165.33,1627.80,NaN,1348.26,0.00,0.0
2,2013-01-01 02:00:00,217.00,25.70,25.70,0.0,0.0,0.2,0.0,44.10,NaN,...,0.00,0.00,4.88,1453.91,170.33,1546.70,NaN,1278.70,0.00,0.0
3,2013-01-01 03:00:00,220.00,25.70,25.70,0.0,0.0,0.2,0.0,43.60,NaN,...,0.00,0.00,0.37,1407.16,170.33,1467.40,NaN,1236.46,0.00,0.0
4,2013-01-01 04:00:00,225.00,25.70,25.70,0.0,0.0,0.2,0.0,44.20,NaN,...,0.00,0.00,0.83,1382.51,170.33,1419.00,NaN,1211.35,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107107,2025-03-21 19:00:00,290.58,18.45,25.99,0.0,0.2,0.2,0.2,49.99,0.00,...,45.05,31.07,84.59,3245.96,364.83,3099.86,0.1,2796.44,20.19,0.0
107108,2025-03-21 20:00:00,280.66,26.64,26.27,0.0,0.2,0.2,0.2,49.14,0.97,...,44.52,32.86,78.30,3278.37,366.18,3104.64,0.0,2833.89,20.15,0.0
107109,2025-03-21 21:00:00,267.93,26.66,26.30,0.0,0.2,0.2,0.2,50.21,3.50,...,43.16,32.68,84.72,3274.15,387.65,3067.59,0.0,2801.78,20.04,0.0
107110,2025-03-21 22:00:00,284.77,26.69,26.34,0.0,0.2,0.2,0.2,49.33,0.44,...,43.32,32.65,80.89,3241.81,365.87,3010.62,0.0,2795.05,20.09,0.0


In [16]:
import requests, torch, pickle
import pandas as pd, numpy as np
from pathlib import Path
import datetime
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_lightning import seed_everything

# ---------------------------
# Funciones de Feature Engineering
# ---------------------------
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df.index.hour
    df["hour_sin"] = np.sin(2 * np.pi * df.hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df.hour / 24)
    df["day_of_week"] = df.index.dayofweek
    df["dow_sin"] = np.sin(2 * np.pi * df.day_of_week / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df.day_of_week / 7)
    df["month"] = df.index.month
    df["month_sin"] = np.sin(2 * np.pi * df.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df.month / 12)
    return df

def add_lag_features(df: pd.DataFrame, cols, lags=[1,2,3]) -> pd.DataFrame:
    df = df.copy()
    new_cols = {}
    for col in cols:
        for lag in lags:
            new_cols[f"{col}_lag{lag}"] = df[col].shift(lag)
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

def add_moving_average_features(df: pd.DataFrame, cols, windows=[3,6]) -> pd.DataFrame:
    df = df.copy()
    new_cols = {}
    for col in cols:
        for w in windows:
            new_cols[f"{col}_ma{w}"] = df[col].rolling(window=w, min_periods=1).mean()
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_temporal_features(df)
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col != "generation"]
    df = add_lag_features(df, numeric_cols)
    df = add_moving_average_features(df, numeric_cols)
    df = df.ffill().bfill()
    return df.dropna()

# ---------------------------
# Funciones para obtener datos de forecast
# ---------------------------
def get_project_coordinates(name: str, csv_path: Path):
    df = pd.read_csv(csv_path, encoding="latin1")
    row = df[df["CENTRAL"].str.lower() == name.lower()]
    return (float(row.Latitud.iloc[0]), float(row.Longitud.iloc[0])) if not row.empty else None

def get_forecast(lat: float, lon: float, days: int = 2) -> pd.DataFrame:
    WEATHER_VARS = [
        "temperature_2m", "wind_speed_10m", "relative_humidity_2m", "wind_gusts_10m",
        "vapour_pressure_deficit", "cloud_cover", "cloud_cover_low", "cloud_cover_mid",
        "cloud_cover_high", "surface_pressure", "pressure_msl", "apparent_temperature",
        "rain", "shortwave_radiation", "diffuse_radiation", "global_tilted_irradiance",
        "shortwave_radiation_instant", "diffuse_radiation_instant", "global_tilted_irradiance_instant",
        "direct_radiation", "direct_normal_irradiance", "terrestrial_radiation",
        "direct_radiation_instant", "direct_normal_irradiance_instant", "terrestrial_radiation_instant"
    ]
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join(WEATHER_VARS),
        "forecast_days": days,
        "timezone": "UTC"
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()["hourly"]
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df.pop("time"), utc=True)
    return df.set_index("date")

# ---------------------------
# Main: Proceso completo de obtención y predicción
# ---------------------------
def main():
    # Rutas
    CSV_INFO         = Path("../data/lookup/central_info.csv")
    MODEL_CLEAN_PATH = Path("../data/interim/meteo_data_with_generation/parque_solar_girasol.parquet")
    MODEL_PATH       = Path("../data/models/tft/best_tft_model.pth")
    OUTPUT_PATH      = Path("../data/processed_predictions/parque_solar_girasol_forecast_predictions.parquet")
    
    seed_everything(42)
    
    # 1. Cargar datos históricos para reconstruir la estructura del dataset
    df_hist = pd.read_parquet(MODEL_CLEAN_PATH)
    if "date" in df_hist.columns:
        df_hist.index = pd.to_datetime(df_hist["date"], utc=True)
        df_hist.drop(columns=["date"], inplace=True)
    else:
        df_hist.index = pd.to_datetime(df_hist.index, utc=True)
    
    # Aplicar feature engineering a los datos históricos (como en entrenamiento)
    df_hist = add_temporal_features(df_hist)
    weather_vars = [
        "temperature_2m", "wind_speed_10m", "relative_humidity_2m", "wind_gusts_10m",
        "vapour_pressure_deficit", "cloud_cover", "cloud_cover_low", "cloud_cover_mid",
        "cloud_cover_high", "surface_pressure", "pressure_msl", "apparent_temperature",
        "rain", "shortwave_radiation", "diffuse_radiation", "global_tilted_irradiance",
        "shortwave_radiation_instant", "diffuse_radiation_instant", "global_tilted_irradiance_instant",
        "direct_radiation", "direct_normal_irradiance", "terrestrial_radiation",
        "direct_radiation_instant", "direct_normal_irradiance_instant", "terrestrial_radiation_instant"
    ]
    numeric_cols = [col for col in df_hist.columns if col != "generation" and pd.api.types.is_numeric_dtype(df_hist[col])]
    df_hist = add_lag_features(df_hist, numeric_cols)
    df_hist = add_moving_average_features(df_hist, numeric_cols)
    df_hist = df_hist.ffill().bfill()
    df_hist["time_idx"] = ((df_hist.index - df_hist.index.min()).total_seconds() // 3600).astype(int)
    df_hist["group_id"] = "plant_1"
    
    # Crear el TimeSeriesDataSet de entrenamiento (igual que en entrenamiento)
    training_dataset = TimeSeriesDataSet(
        df_hist,
        time_idx="time_idx",
        target="generation",
        group_ids=["group_id"],
        min_encoder_length=12,
        max_encoder_length=12,
        min_prediction_length=12,
        max_prediction_length=12,
        time_varying_known_reals=["time_idx"] + weather_vars +
            ["hour", "hour_sin", "hour_cos", "day_of_week", "dow_sin", "dow_cos", "month", "month_sin", "month_cos"],
        time_varying_unknown_reals=["generation"] +
            [col for col in df_hist.columns if col not in weather_vars + ["generation", "time_idx", "group_id",
                                                                          "hour", "hour_sin", "hour_cos",
                                                                          "day_of_week", "dow_sin", "dow_cos",
                                                                          "month", "month_sin", "month_cos"]],
        target_normalizer=GroupNormalizer(groups=["group_id"], transformation="softplus"),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True,
        randomize_length=False
    )
    
    # 2. Obtener datos de forecast desde la API (solicitando 2 días)
    coords = get_project_coordinates("parque solar girasol", CSV_INFO) or (18.333668, -70.198569)
    df_forecast_api = get_forecast(*coords, days=2)
    # Ajustar 4 horas hacia atrás (API adelantada)
    df_forecast_api.index -= pd.Timedelta(hours=4)
    
    unique_days = df_forecast_api.index.normalize().unique()
    if len(unique_days) < 2:
        raise ValueError("No se obtuvieron datos de dos días. Verifica la respuesta de la API.")
    hist_day = unique_days[0]
    fcst_day = unique_days[1]
    
    df_hist_api = df_forecast_api.loc[df_forecast_api.index.normalize() == hist_day]
    df_fcst_api = df_forecast_api.loc[df_forecast_api.index.normalize() == fcst_day]
    
    # 3. Concatenar el día histórico con el de forecast para calcular lags y medias móviles
    df_combined = pd.concat([df_hist_api, df_fcst_api])
    df_combined_feat = add_features(df_combined)
    # Extraer únicamente la porción correspondiente al día de forecast
    df_fcst_feat = df_combined_feat.loc[df_fcst_api.index.min(): df_fcst_api.index.max()].copy()
    
    # Como en forecast no se cuenta con generación histórica, se asigna 0.0 (float) a "generation"
    df_fcst_feat["generation"] = 0.0
    df_fcst_feat["group_id"] = "plant_1"
    # Actualizar time_idx a partir del mínimo de los datos históricos para mantener la escala
    df_fcst_feat["time_idx"] = ((df_fcst_feat.index - df_hist.index.min()).total_seconds() // 3600).astype(int)
    
    # Guardamos el índice del DataFrame de forecast para reconstruir las predicciones
    forecast_index = df_fcst_feat.index
    
    # 4. Crear el dataset de predicción a partir del dataset de entrenamiento
    prediction_dataset = training_dataset.from_dataset(training_dataset, df_fcst_feat, predict_mode=True)
    
    # 5. Instanciar el modelo TFT usando los hiperparámetros exactos del entrenamiento (hidden_size=16, attention_heads=1, etc.)
    best_params = {
        "learning_rate": 0.001,
        "hidden_size": 16,
        "attention_heads": 1,
        "dropout": 0.1
    }
    model = TemporalFusionTransformer.from_dataset(
        training_dataset,
        learning_rate=best_params["learning_rate"],
        hidden_size=best_params["hidden_size"],
        attention_head_size=best_params["attention_heads"],
        dropout=best_params["dropout"],
        hidden_continuous_size=best_params["hidden_size"],
        output_size=1,
        loss=QuantileLoss(quantiles=[0.5]),  # Usar únicamente el cuantíl 0.5 para inferencia
        log_interval=100,
        reduce_on_plateau_patience=4,
    )
    
    # Cargar el state dict del modelo entrenado
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device("cpu")))
    model.eval()
    
    # 6. Realizar la predicción
    predict_dataloader = prediction_dataset.to_dataloader(train=False, batch_size=64)
    predictions = model.predict(predict_dataloader)
    
    # 7. Guardar las predicciones usando el índice del DataFrame de forecast
    df_preds = pd.DataFrame(predictions, index=forecast_index, columns=["prediction"])
    df_preds.to_parquet(OUTPUT_PATH)
    print("✅ Predicciones guardadas en", OUTPUT_PATH)

if __name__ == "__main__":
    main()

Seed set to 42
c:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\my_environment\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\my_environment\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
c:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\my_environment\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'p

ValueError: Shape of passed values is (1, 12), indices imply (24, 1)